# Generic Preprocessing

Processes datasets for the PRiSM framework, supporting OpenML datasets and CSV files.

## Quick Start

1. **Configure your dataset** in `.env` (copy from `.env.example`):
   ```bash
   # Option 1: Full config mode (recommended)
   PRISM_CONFIG=htx_example    # Loads example_notebooks/config/htx_example.yaml

   # Option 2: Quick mode (no YAML needed)
   PRISM_DATASET=my_data       # Uses notebook defaults, loads data/raw/my_data.csv
   ```

2. **Place your data** in `data/raw/` (e.g., `data/raw/my_data.csv`)

3. **Run this notebook** to preprocess and split your data

## Configuration

**Priority**: `.env` file → YAML config → Manual settings in notebook

YAML configs in `example_notebooks/config/` can specify:
- Target/ID columns, splitting method, encoding strategy
- Ordinal feature orderings, feature selection, etc.

See `example_notebooks/config/example_config.yaml` for all options.

## Setup and Configuration

In [ ]:
%load_ext autoreload
%autoreload 2

# Set CPU fallback for MPS (Apple Silicon)
import os

os.environ['PYTORCH_ENABLE_MPS_FALLBACK'] = '1'

# Core imports
import numpy as np
import pandas as pd
import logging
import matplotlib.pyplot as plt
from pathlib import Path
from datetime import datetime

# Custom modules
from prism.config import RAW_DATA_DIR, INTERIM_DATA_DIR, PROCESSED_DATA_DIR, DATASET_PREFIX, PROJ_ROOT, CONFIG_NAME
from prism.logging_config import setup_logging
from prism.config_loader import (
    load_config_if_exists,
    DEFAULT_TARGET_CANDIDATES,
    DEFAULT_ID_CANDIDATES,
)

# =============================================================================
# DATASET CHECK - Ensure .env is configured
# =============================================================================
if DATASET_PREFIX is None:
    print("=" * 70)
    print("ERROR: No dataset configured!")
    print("=" * 70)
    print("\nPlease configure your dataset in the .env file:")
    print("  1. Copy .env.example to .env")
    print("  2. Set PRISM_CONFIG=your_config (loads example_notebooks/config/your_config.yaml)")
    print("     OR set PRISM_DATASET=your_dataset (quick mode, no YAML needed)")
    print("\nExample .env contents:")
    print("  PRISM_CONFIG=htx_example")
    print("=" * 70)
    raise ValueError("DATASET_PREFIX is None - configure .env file")

print(f"Dataset: {DATASET_PREFIX}")
if CONFIG_NAME:
    print(f"Config: {CONFIG_NAME}.yaml")
from prism.data_utilities import (
    split_data_temporal_or_random,
    save_split_datasets,
    load_data,
    enforce_binary_target_encoding,
    convert_to_categorical,
    analyze_categorical_columns,
    create_binary_encoding,
    create_ordinal_encoding,
    ensure_consistent_categories,
)
from prism.preprocessing import preprocess_data

# Conditional import for OpenML (only needed for OpenML datasets)
try:
    import openml

    OPENML_AVAILABLE = True
except ImportError:
    OPENML_AVAILABLE = False
    print("Warning: openml package not available. OpenML datasets will not be loadable.")

## Configuration

Configuration can be provided in two ways:
1. **Config files** (recommended): Create `{DATASET_PREFIX}.yaml` in `example_notebooks/config/`
2. **Manual settings**: Modify the variables below directly

Config files will override the manual settings below if they exist.

### Available Configuration Options in YAML files:

**General:**
- `random_seed`: Random seed for reproducibility (int)

**Categorical Encoding:**
- `integer_encoding`: Dict mapping feature names to ordered category lists
- `force_all_ordinal`: Force all categorical features to be ordinal encoded (bool)
- `convert_numeric`: Apply categorical threshold to numeric variables (bool)
- `categorical_threshold`: Max unique values for numeric→categorical conversion (int)

**Feature Selection:**
- `selected_features`: List of feature names to include (empty list = use all)

**Data Splitting:**
- `splitting_method`: 'random' or 'temporal' (str)
- `split_ratios`: [train, test, validation] ratios as list of floats
- `time_variable`: Column name for temporal splitting (str)

**Target & ID Variables:**
- `target_candidates`: List of potential target column names (list)
- `id_candidates`: List of potential ID column names (list)
- `invert_target`: Flip target values 0↔1 (bool)

See `example_config.yaml` for a comprehensive example.

In [ ]:
# ============================================================================
# PROJECT CONFIGURATION
# ============================================================================
random_seed = 257
PROJECT_NAME = DATASET_PREFIX
np.random.seed(random_seed)

# ============================================================================
# DATA SOURCE CONFIGURATION
# ============================================================================
# Data source is auto-detected from DATASET_PREFIX:
#   - Format "openml_<id>" → Load from OpenML API
#   - Any other format → Load from CSV file "{DATASET_PREFIX}.csv"
DATA_FILENAME = f"{DATASET_PREFIX}.csv"

# ============================================================================
# CATEGORICAL ENCODING CONFIGURATION (MANUAL DEFAULTS)
# ============================================================================
# Categorical variable detection
CONVERT_NUMERIC = False  # Apply categorical threshold to numeric variables
CATEGORICAL_THRESHOLD = 12  # Variables with fewer unique values treated as categorical

# Encoding strategy for categorical variables
# Logic precedence: INTEGER_ENCODING (if provided) → FORCE_ALL_ORDINAL → default behavior
FORCE_ALL_ORDINAL = False  # If True: ALL categorical → ordinal
# If False: Binary → ordinal, Multi-category → one-hot (default)

# Advanced: Manual ordinal configuration (overrides automatic detection)
# Set specific ordinal orderings for variables (leave empty {} for automatic)
# Example: {'education': ['elementary', 'high_school', 'college', 'graduate']}
INTEGER_ENCODING = {}

# ============================================================================
# FEATURE SELECTION CONFIGURATION (MANUAL DEFAULTS)
# ============================================================================
# Specify features to include (empty list = use all features)
# Example: ['age', 'duration', 'credit_amount', 'employment']
SELECTED_FEATURES = []

# ============================================================================
# DATA SPLITTING CONFIGURATION (MANUAL DEFAULTS)
# ============================================================================
SPLITTING_METHOD = "random"  # Options: "random", "temporal", or "predefined"
TIME_VARIABLE = (
    "tx_year"  # Column to use for temporal splitting (only used if SPLITTING_METHOD="temporal")
)
SPLIT_COLUMN = None  # Column with predefined split labels (only used if SPLITTING_METHOD="predefined")
DROP_SPLIT_COLUMN = True  # Drop split_column from data after predefined splitting
SPLIT_LABELS = None  # Custom mapping for split labels, e.g., {'train': ['Tr'], 'test': ['Te'], 'val': ['Va']}
SPLIT_RATIOS = (0.6, 0.2, 0.2)  # (train, test, val) ratios (for random/temporal only)
DROP_TIME_VARIABLE = True  # Drop time_variable from data after temporal splitting

# ============================================================================
# TARGET AND ID VARIABLE CONFIGURATION (MANUAL DEFAULTS)
# ============================================================================
# These defaults are imported from prism.config_loader but can be overridden here.
# To customize, uncomment and modify the lines below:
# TARGET_CANDIDATES = ['my_target', 'outcome', 'label']
# ID_CANDIDATES = ['my_id', 'patient_id']
TARGET_CANDIDATES = list(DEFAULT_TARGET_CANDIDATES)  # Copy to allow modification
ID_CANDIDATES = list(DEFAULT_ID_CANDIDATES)  # Copy to allow modification
INVERT_TARGET = False  # Set to True to flip 0->1 and 1->0

# ============================================================================
# REFERENCE COLUMN CONFIGURATION (MANUAL DEFAULTS)
# ============================================================================
# Strategy for selecting reference columns in one-hot encoding:
# Options: 'alphabetical', 'most_common', 'manual'
REFERENCE_COLUMN_STRATEGY = 'alphabetical'
# Manual reference columns (only used if strategy is 'manual')
# Example: {'category_group': 'category_group_RefValue'}
MANUAL_REFERENCE_COLUMNS = None

# ============================================================================
# CONFIG FILE LOADING (OVERWRITES MANUAL SETTINGS ABOVE)
# ============================================================================
# Load configuration from YAML file if it exists
# This will override the manual defaults above if a config file is found
print("\n" + "=" * 70)
print("LOADING CONFIGURATION")
print("=" * 70)

# Use CONFIG_NAME if set (from .env PRISM_CONFIG), otherwise try DATASET_PREFIX
config_to_load = CONFIG_NAME if CONFIG_NAME else DATASET_PREFIX
config = load_config_if_exists(config_to_load)

if config:
    print(f"Config file found: {config_to_load}.yaml")
    print("Applying config file settings (overriding manual defaults):")

    # Load random seed first (affects subsequent random operations)
    if 'random_seed' in config:
        random_seed = config['random_seed']
        np.random.seed(random_seed)
        print(f"   [ok] random_seed: {random_seed}")

    # Categorical encoding config options
    if 'integer_encoding' in config:
        INTEGER_ENCODING = config['integer_encoding']
        print(f"   [ok] integer_encoding: {list(INTEGER_ENCODING.keys())}")

    if 'force_all_ordinal' in config:
        FORCE_ALL_ORDINAL = config['force_all_ordinal']
        print(f"   [ok] force_all_ordinal: {FORCE_ALL_ORDINAL}")

    if 'convert_numeric' in config:
        CONVERT_NUMERIC = config['convert_numeric']
        print(f"   [ok] convert_numeric: {CONVERT_NUMERIC}")

    if 'categorical_threshold' in config:
        CATEGORICAL_THRESHOLD = config['categorical_threshold']
        print(f"   [ok] categorical_threshold: {CATEGORICAL_THRESHOLD}")

    # Feature selection configuration
    if 'selected_features' in config:
        SELECTED_FEATURES = config['selected_features']
        print(f"   [ok] selected_features: {len(SELECTED_FEATURES)} features")

    # Data splitting configuration
    if 'splitting_method' in config:
        SPLITTING_METHOD = config['splitting_method']
        print(f"   [ok] splitting_method: {SPLITTING_METHOD}")

    if 'time_variable' in config:
        TIME_VARIABLE = config['time_variable']
        print(f"   [ok] time_variable: {TIME_VARIABLE}")

    if 'split_ratios' in config:
        SPLIT_RATIOS = tuple(config['split_ratios'])
        print(f"   [ok] split_ratios: {SPLIT_RATIOS}")

    if 'drop_time_variable' in config:
        DROP_TIME_VARIABLE = config['drop_time_variable']
        print(f"   [ok] drop_time_variable: {DROP_TIME_VARIABLE}")

    # Predefined split configuration
    if 'split_column' in config:
        SPLIT_COLUMN = config['split_column']
        print(f"   [ok] split_column: {SPLIT_COLUMN}")

    if 'drop_split_column' in config:
        DROP_SPLIT_COLUMN = config['drop_split_column']
        print(f"   [ok] drop_split_column: {DROP_SPLIT_COLUMN}")

    if 'split_labels' in config:
        SPLIT_LABELS = config['split_labels']
        print(f"   [ok] split_labels: configured for {list(SPLIT_LABELS.keys())}")

    # Target and ID variable configuration
    if 'target_candidates' in config:
        TARGET_CANDIDATES = config['target_candidates']
        print(f"   [ok] target_candidates: {TARGET_CANDIDATES}")

    if 'id_candidates' in config:
        ID_CANDIDATES = config['id_candidates']
        print(f"   [ok] id_candidates: {ID_CANDIDATES}")

    if 'invert_target' in config:
        INVERT_TARGET = config['invert_target']
        print(f"   [ok] invert_target: {INVERT_TARGET}")

    if 'reference_column_strategy' in config:
        REFERENCE_COLUMN_STRATEGY = config['reference_column_strategy']
        print(f"   [ok] reference_column_strategy: {REFERENCE_COLUMN_STRATEGY}")

    if 'reference_columns' in config:
        MANUAL_REFERENCE_COLUMNS = config['reference_columns']
        print(f"   [ok] reference_columns: {list(MANUAL_REFERENCE_COLUMNS.keys())}")

    print("Config loading complete!")
else:
    print("No config file found - using manual settings above")
    print("Edit the settings in the cells above if needed.")

print("=" * 70)

In [ ]:
# ============================================================================
# CONFIGURATION SUMMARY
# ============================================================================
print("\n" + "=" * 70)
print("FINAL CONFIGURATION SUMMARY")
print("=" * 70)
print(f"Dataset prefix: {DATASET_PREFIX}")
print(f"Project name: {PROJECT_NAME}")
print(f"Random seed: {random_seed}")
if config:
    config_count = len(config)
    print(f"Config file applied: {config_count} settings from {DATASET_PREFIX}.yaml")
else:
    print(f"Config file: None found - using all manual settings")

print(f"\nData Source:")
print(f"  Auto-detection from DATASET_PREFIX format")
if DATASET_PREFIX.startswith('openml_'):
    print(f"  → Detected: OpenML dataset")
else:
    print(f"  → Detected: CSV file ({DATA_FILENAME})")

print(f"\nCategorical Encoding:")
print(f"  Convert numeric: {CONVERT_NUMERIC}")
print(f"  Categorical threshold: {CATEGORICAL_THRESHOLD} unique values")
if INTEGER_ENCODING:
    strategy_source = "from config file" if config and 'integer_encoding' in config else "manual"
    print(f"  Strategy: Ordinal configuration ({strategy_source})")
    print(f"  Ordinal features: {list(INTEGER_ENCODING.keys())}")
elif FORCE_ALL_ORDINAL:
    print(f"  Strategy: Force all categorical → ordinal")
else:
    print(f"  Strategy: Default (Binary → integer, Multi-category → one-hot)")

print(f"\nFeature Selection:")
if SELECTED_FEATURES:
    selection_source = "from config file" if config and 'selected_features' in config else "manual"
    print(f"  Using selected features: {len(SELECTED_FEATURES)} features ({selection_source})")
else:
    print(f"  Using all features")

print(f"\nData Splitting:")
splitting_source = "from config file" if config and 'splitting_method' in config else "manual"
print(f"  Method: {SPLITTING_METHOD} ({splitting_source})")
if SPLITTING_METHOD == "temporal":
    time_source = "from config file" if config and 'time_variable' in config else "manual"
    print(f"  Time variable: {TIME_VARIABLE} ({time_source})")
print(
    f"  Split ratios (train:test:val): {SPLIT_RATIOS[0]:.1f}:{SPLIT_RATIOS[1]:.1f}:{SPLIT_RATIOS[2]:.1f}"
)

print(f"\nID Variable:")
id_source = "from config file" if config and 'id_candidates' in config else "manual"
print(f"  Candidates: {ID_CANDIDATES} ({id_source})")

print(f"\nTarget Variable:")
target_source = "from config file" if config and 'target_candidates' in config else "manual"
print(f"  Candidates: {TARGET_CANDIDATES} ({target_source})")
invert_source = "from config file" if config and 'invert_target' in config else "manual"
print(f"  Invert target: {INVERT_TARGET} ({invert_source})")

print(f"\nOutput:")
print(f"  Files will be saved to: {INTERIM_DATA_DIR}")
print("=" * 70)

In [ ]:
# Set up logging
timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")

setup_logging(
    file_log_level=logging.INFO,
    log_file=f'preprocessing_{DATASET_PREFIX}_{timestamp}.log',
    console_output=True,
    root_log_level=logging.WARNING,
)

## Data Loading (Auto-Detection)

In [ ]:
# Detect data source from DATASET_PREFIX and load accordingly
print("\n" + "=" * 70)
print("DATA LOADING")
print("=" * 70)

# Check if this is an OpenML dataset
if DATASET_PREFIX.startswith('openml_'):
    # ========================================================================
    # OPENML DATA LOADING
    # ========================================================================
    if not OPENML_AVAILABLE:
        raise ImportError("OpenML package not available. Install with: pip install openml")

    # Extract dataset ID from DATASET_PREFIX (e.g., "openml_31" -> 31)
    try:
        dataset_id = int(DATASET_PREFIX.split('_')[-1])
        print(f"Loading from OpenML...")
        print(f"Dataset ID: {dataset_id}")
    except (ValueError, IndexError):
        raise ValueError(
            f"Invalid DATASET_PREFIX format: {DATASET_PREFIX}. " f"Expected format: 'openml_<id>'"
        )

    try:
        # Fetch dataset from OpenML
        dataset = openml.datasets.get_dataset(dataset_id)

        # Get the data as pandas DataFrame
        X, y, categorical_indicator, attribute_names = dataset.get_data(
            dataset_format="dataframe", target=dataset.default_target_attribute
        )

        # Combine features and target into single DataFrame
        if y is not None:
            # Add target variable to DataFrame
            target_name = dataset.default_target_attribute
            df_loaded = X.copy()
            df_loaded[target_name] = y
            print(f"  Target variable: {target_name}")
        else:
            df_loaded = X.copy()
            target_name = None
            print("  No target variable found in dataset")

        # Display dataset information
        print(f"\nOpenML Dataset: {dataset.name}")
        print(
            f"Description: {dataset.description[:200]}..."
            if len(dataset.description) > 200
            else f"Description: {dataset.description}"
        )
        print(f"Dataset shape: {df_loaded.shape[0]} rows, {df_loaded.shape[1]} columns")

        # Rename the target column to 'target' for consistency
        if target_name and target_name in df_loaded.columns:
            df_loaded = df_loaded.rename(columns={target_name: 'target'})
            print(f"Renamed target column from '{target_name}' to 'target'")

    except Exception as e:
        print(f"Error loading dataset from OpenML: {e}")
        print(f"Please check that dataset ID {dataset_id} exists on OpenML")
        raise

else:
    # ========================================================================
    # CSV DATA LOADING
    # ========================================================================
    print(f"Loading from CSV file...")
    print(f"Data directory: {RAW_DATA_DIR}")
    print(f"Filename: {DATA_FILENAME}")

    # Load the dataset using prism.data_utilities.load_data
    df_loaded = load_data(DATA_FILENAME)
    print(f"Dataset shape: {df_loaded.shape[0]} rows, {df_loaded.shape[1]} columns")

# Display basic information
print(f"\nDataset columns: {list(df_loaded.columns)}")
print(f"\nBasic statistics:")
print(f"  Numerical columns: {df_loaded.select_dtypes(include=[np.number]).shape[1]}")
print(f"  Non-numerical columns: {df_loaded.select_dtypes(exclude=[np.number]).shape[1]}")
print(f"  Missing values: {df_loaded.isnull().sum().sum()}")

# Print unique values for object/categorical type columns
obj_cat_cols = df_loaded.select_dtypes(include=['object', 'category']).columns
if len(obj_cat_cols) > 0 and len(obj_cat_cols) <= 10:  # Only print if reasonable number
    print(f"\nUnique values for object/categorical columns ({len(obj_cat_cols)} columns):")
    for col in obj_cat_cols:
        unique_vals = df_loaded[col].unique()
        if len(unique_vals) <= 20:  # Only print if not too many values
            print(f"  {col}: {list(unique_vals)}")
        else:
            print(f"  {col}: {len(unique_vals)} unique values")

print("=" * 70)

## Target Variable Detection and Encoding

In [ ]:
def detect_target_variable(df, target_candidates):
    """Detect target variable from candidates."""
    for candidate in target_candidates:
        if candidate in df.columns:
            return candidate
    return None


# Detect target variable
target_variable = detect_target_variable(df_loaded, TARGET_CANDIDATES)
if target_variable:
    print(f"Detected target variable: {target_variable}")
else:
    print("Warning: No target variable detected from candidates")

In [ ]:
# Apply binary encoding to target variable
if target_variable:
    print(f"\nApplying binary target encoding...")
    df_loaded = enforce_binary_target_encoding(df_loaded, target_variable)
    print(f"Binary encoding applied to '{target_variable}'")
    print(f"Final value counts:\n{df_loaded[target_variable].value_counts().sort_index()}")

    # Optionally invert target variable
    if INVERT_TARGET:
        print(f"\nInverting target variable '{target_variable}'...")
        original_counts = df_loaded[target_variable].value_counts().sort_index()
        print(f"Original distribution: {dict(original_counts)}")

        df_loaded[target_variable] = 1 - df_loaded[target_variable]

        inverted_counts = df_loaded[target_variable].value_counts().sort_index()
        print(f"Inverted distribution: {dict(inverted_counts)}")
        print("Target variable successfully inverted.")
    else:
        print(f"Target variable kept as-is (INVERT_TARGET = False)")
else:
    print("No target variable detected - skipping binary encoding")

print("=" * 70)

## ID Column Detection and Removal

In [ ]:
def detect_id_variable(df, id_candidates):
    """Detect id variable from candidates."""
    for candidate in id_candidates:
        if candidate in df.columns:
            return candidate
    return None


# Detect id variable
id_variable = detect_id_variable(df_loaded, ID_CANDIDATES)
if id_variable:
    print(f"Detected ID variable: {id_variable}")
    print(f"Total values: {len(df_loaded[id_variable])}")
    print(f"Unique values: {df_loaded[id_variable].nunique()}")
    if df_loaded[id_variable].nunique() == len(df_loaded):
        print("[ok] All values are unique")
    else:
        duplicates = len(df_loaded) - df_loaded[id_variable].nunique()
        print(f"Warning: {duplicates} duplicate values found")

    # Drop the ID column
    df_no_id = df_loaded.drop(columns=[id_variable])
else:
    print("No ID variable detected from candidates")
    print(f"Candidates searched: {ID_CANDIDATES}")
    print("A synthetic ID will be created later in the pipeline")
    df_no_id = df_loaded

## Categorical Variable Detection and Analysis

In [ ]:
print("\n" + "=" * 70)
print("CATEGORICAL VARIABLE DETECTION")
print("=" * 70)

# Determine columns to exclude from categorical conversion
# Exclude temporal columns to preserve them as numeric for temporal splitting
# Exclude split columns to preserve them as-is for predefined splitting
exclude_from_categorical = []
if SPLITTING_METHOD == "temporal" and TIME_VARIABLE in df_no_id.columns:
    exclude_from_categorical.append(TIME_VARIABLE)
    print(f"Excluding temporal column '{TIME_VARIABLE}' from categorical conversion")
if SPLITTING_METHOD == "predefined" and SPLIT_COLUMN and SPLIT_COLUMN in df_no_id.columns:
    exclude_from_categorical.append(SPLIT_COLUMN)
    print(f"Excluding split column '{SPLIT_COLUMN}' from categorical conversion")

# Convert to categorical based on configuration
df_processed = convert_to_categorical(
    df_no_id,
    categorical_threshold=CATEGORICAL_THRESHOLD,
    convert_numeric=CONVERT_NUMERIC,
    exclude_columns=exclude_from_categorical,
)

# Analyze categorical columns
categorical_info = analyze_categorical_columns(df_processed)

binary_vars = categorical_info['binary']
multi_cat_vars = categorical_info['multi_category']

print("=" * 70)

## Optional Feature Selection

In [ ]:
if SELECTED_FEATURES:
    print("\n" + "=" * 70)
    print("FEATURE SELECTION")
    print("=" * 70)
    print(f"Filtering to {len(SELECTED_FEATURES)} selected features...")

    # Always include target variable and ID candidates if they exist
    required_cols = []
    if target_variable and target_variable in df_processed.columns:
        required_cols.append(target_variable)
    for id_candidate in ID_CANDIDATES:
        if id_candidate in df_processed.columns:
            required_cols.append(id_candidate)

    # Combine selected features with required columns
    cols_to_keep = list(set(SELECTED_FEATURES + required_cols))

    # Filter to available columns
    available_cols = [col for col in cols_to_keep if col in df_processed.columns]
    missing_cols = [col for col in cols_to_keep if col not in df_processed.columns]

    if missing_cols:
        print(f"Warning: {len(missing_cols)} selected features not found in dataset:")
        print(f"  {missing_cols}")

    print(f"Keeping {len(available_cols)} columns:")
    print(f"  Selected features: {len([c for c in available_cols if c in SELECTED_FEATURES])}")
    print(f"  Required columns: {len([c for c in available_cols if c not in SELECTED_FEATURES])}")

    # Apply filtering
    df_processed = df_processed[available_cols]

    # Update categorical variable lists to only include selected features
    binary_vars = [v for v in binary_vars if v in available_cols]
    multi_cat_vars = [v for v in multi_cat_vars if v in available_cols]

    print(f"\nAfter filtering:")
    print(f"  Binary categorical: {len(binary_vars)}")
    print(f"  Multi-category categorical: {len(multi_cat_vars)}")
    print("=" * 70)

## Data Splitting

In [ ]:
print("\n" + "=" * 70)
print("DATA SPLITTING")
print("=" * 70)
print(f"Splitting method: {SPLITTING_METHOD}")

if SPLITTING_METHOD == "predefined":
    from prism.data_utilities import split_data_predefined

    if not SPLIT_COLUMN:
        raise ValueError(
            "Predefined splitting requested but 'split_column' is not configured.\n"
            "Set SPLIT_COLUMN in the notebook or add 'split_column' to your YAML config file.\n"
            "Example config:\n"
            "  splitting_method: 'predefined'\n"
            "  split_column: 'split'"
        )
    if SPLIT_COLUMN not in df_processed.columns:
        raise ValueError(
            f"Predefined splitting requested but split column '{SPLIT_COLUMN}' not found in data.\n"
            f"Available columns: {list(df_processed.columns)}\n"
            "Check that the column name matches exactly (case-sensitive)."
        )
    print(f"Performing predefined split based on column: {SPLIT_COLUMN}")
    train_df_raw, test_df_raw, val_df_raw = split_data_predefined(
        df_processed,
        split_column=SPLIT_COLUMN,
        split_labels=SPLIT_LABELS,
    )
elif SPLITTING_METHOD == "temporal":
    if TIME_VARIABLE not in df_processed.columns:
        raise ValueError(
            f"Temporal splitting requested but time variable '{TIME_VARIABLE}' not found in data.\n"
            f"Available columns: {list(df_processed.columns)}\n"
            "Set TIME_VARIABLE in the notebook or add 'time_variable' to your YAML config file."
        )
    print(f"Performing temporal split based on column: {TIME_VARIABLE}")
    train_df_raw, test_df_raw, val_df_raw = split_data_temporal_or_random(
        df_processed,
        temporal_column=TIME_VARIABLE,
        train_size=SPLIT_RATIOS[0],
        test_size=SPLIT_RATIOS[1],
        val_size=SPLIT_RATIOS[2],
        stratify=df_processed[target_variable] if target_variable else None,
        random_state=random_seed,
    )
else:
    print("Performing random split, stratified by target variable")
    train_df_raw, test_df_raw, val_df_raw = split_data_temporal_or_random(
        df_processed,
        temporal_column=None,  # NB when None, split is random
        train_size=SPLIT_RATIOS[0],
        test_size=SPLIT_RATIOS[1],
        val_size=SPLIT_RATIOS[2],
        stratify=df_processed[target_variable] if target_variable else None,
        random_state=random_seed,
    )

# Drop split column from all splits if configured and present
if SPLITTING_METHOD == "predefined" and DROP_SPLIT_COLUMN:
    if SPLIT_COLUMN and SPLIT_COLUMN in train_df_raw.columns:
        print(f"Dropping split column '{SPLIT_COLUMN}' from all splits")
        train_df_raw = train_df_raw.drop(columns=[SPLIT_COLUMN])
        test_df_raw = test_df_raw.drop(columns=[SPLIT_COLUMN])
        val_df_raw = val_df_raw.drop(columns=[SPLIT_COLUMN])
    else:
        print(f"Notice: Split column '{SPLIT_COLUMN}' not found for dropping")

# Drop time variable from all splits if configured and present
if SPLITTING_METHOD == "temporal" and DROP_TIME_VARIABLE:
    if TIME_VARIABLE in train_df_raw.columns:
        print(f"Dropping temporal column '{TIME_VARIABLE}' from all splits")
        train_df_raw = train_df_raw.drop(columns=[TIME_VARIABLE])
        test_df_raw = test_df_raw.drop(columns=[TIME_VARIABLE])
        val_df_raw = val_df_raw.drop(columns=[TIME_VARIABLE])
    else:
        print(f"Notice: Temporal column '{TIME_VARIABLE}' not found for dropping")

print(f"\nData split completed:")
print(
    f"  Training: {train_df_raw.shape[0]} samples ({train_df_raw.shape[0]/len(df_processed)*100:.1f}%)"
)
print(
    f"  Test: {test_df_raw.shape[0]} samples ({test_df_raw.shape[0]/len(df_processed)*100:.1f}%)"
)
print(
    f"  Validation: {val_df_raw.shape[0]} samples ({val_df_raw.shape[0]/len(df_processed)*100:.1f}%)"
)
print("=" * 70)

## Create Categorical Encoding Mappings

In [ ]:
print("\n" + "=" * 70)
print("CATEGORICAL ENCODING STRATEGY")
print("=" * 70)

# Determine encoding strategy based on configuration
# Logic precedence: INTEGER_ENCODING (if provided) → FORCE_ALL_ORDINAL → default behavior

if INTEGER_ENCODING:
    # ========================================================================
    # MANUAL ORDINAL CONFIGURATION
    # ========================================================================
    print("Strategy: Manual ordinal configuration")
    print(f"Using manually configured ordinal encodings for {len(INTEGER_ENCODING)} variables")

    all_integer_mappings = INTEGER_ENCODING.copy()
    categorical_cols_for_onehot = []

    print(f"\nManually configured ordinal variables:")
    for var, order in all_integer_mappings.items():
        print(f"  {var}: {order}")

    all_categories = {}

elif FORCE_ALL_ORDINAL:
    # ========================================================================
    # FORCE ALL ORDINAL
    # ========================================================================
    print("Strategy: Force all categorical → ordinal")

    # Create ordinal mappings for all categorical variables
    binary_ordinal_mappings = create_binary_encoding(df_processed, binary_vars)
    multi_cat_ordinal_mappings = create_ordinal_encoding(
        df_processed, multi_cat_vars, ordering_method='alphabetical'
    )

    # Combine all ordinal mappings
    all_integer_mappings = {**binary_ordinal_mappings, **multi_cat_ordinal_mappings}
    categorical_cols_for_onehot = []

    print(f"\nAll categorical variables will be ordinal encoded:")
    print(f"  Total: {len(all_integer_mappings)} variables")

    all_categories = {}

else:
    # ========================================================================
    # DEFAULT (Binary → integer, Multi-category → one-hot)
    # ========================================================================
    print("Strategy: default (Binary → ineger, Multi-category → one-hot)")

    # Create ordinal mappings only for binary variables
    binary_ordinal_mappings = create_binary_encoding(df_processed, binary_vars)

    all_integer_mappings = binary_ordinal_mappings

    # Exclude target variable from one-hot encoding
    categorical_cols_for_onehot = [col for col in multi_cat_vars if col != target_variable]

    print(f"\nBinary variables (ordinal encoded): {len(binary_vars)}")
    print(f"Multi-category variables (one-hot encoded): {len(categorical_cols_for_onehot)}")

    # Get all unique values for each one-hot categorical column across all splits
    all_categories = {}
    for col in categorical_cols_for_onehot:
        all_values = set()
        all_values.update(train_df_raw[col].dropna().unique())
        all_values.update(test_df_raw[col].dropna().unique())
        all_values.update(val_df_raw[col].dropna().unique())
        all_categories[col] = sorted(list(all_values))
        print(f"  Column '{col}': {len(all_categories[col])} unique values across all splits")

# Ensure consistent categorical representations across splits
print("\nEnsuring consistent categorical representations across splits...")
train_df_consistent = ensure_consistent_categories(
    train_df_raw, all_categories, all_integer_mappings
)
test_df_consistent = ensure_consistent_categories(
    test_df_raw, all_categories, all_integer_mappings
)
val_df_consistent = ensure_consistent_categories(val_df_raw, all_categories, all_integer_mappings)

print(f"\nPreprocessing strategy summary:")
if INTEGER_ENCODING:
    print(f"  Manual ordinal encoding: {len(all_integer_mappings)} variables")
elif FORCE_ALL_ORDINAL:
    print(f"  All categorical encoding: {len(all_integer_mappings)} ordinal features")
else:
    onehot_feature_count = sum(len(cats) for cats in all_categories.values())
    # Calculate actual number of features after dropping reference columns (one per variable)
    onehot_feature_count_after_drop = onehot_feature_count - len(categorical_cols_for_onehot)
    print(
        f"  One-hot encoding: {len(categorical_cols_for_onehot)} variables → {onehot_feature_count} binary features"
    )
    print(
        f"    (After dropping reference columns: {onehot_feature_count_after_drop} binary features)"
    )
    print(
        f"  Ordinal encoding: {len(all_integer_mappings)} variables → {len(all_integer_mappings)} numeric features"
    )

print("=" * 70)

## Apply Preprocessing to Each Split

In [ ]:
print("\n" + "=" * 70)
print("APPLYING PREPROCESSING")
print("=" * 70)
print("Preprocessing configuration:")
print(
    f"  Ordinal features: {list(all_integer_mappings.keys()) if all_integer_mappings else 'None'}"
)
print(f"  Categorical imputation: random")
print(f"  Numerical imputation: random")
print(f"  Random seed: {random_seed}")

# Construct input file path for hash tracking
input_file_path = RAW_DATA_DIR / DATA_FILENAME

# Process training data
print("\n1. Processing TRAINING data...")
train_processed, train_metadata = preprocess_data(
    train_df_consistent,
    integer_encoding=all_integer_mappings,
    categorical_imputation_method='random',
    numerical_imputation_method='random',
    random_state=random_seed,
    reference_column_strategy=REFERENCE_COLUMN_STRATEGY,
    manual_reference_columns=MANUAL_REFERENCE_COLUMNS,
    drop_reference_columns=True,
    dataset_prefix=DATASET_PREFIX,
    input_file_path=input_file_path,
)
print(f"   Training data processed: {train_processed.shape}")

# Extract reference columns from training metadata for consistency across splits.
# This ensures test/val use the same reference columns as training, which is critical
# when using 'most_common' strategy where different splits might otherwise select
# different reference columns based on their distributions.
train_reference_cols = train_metadata.get('reference_columns', {}).get('references', {})

# Process test data - use 'manual' strategy with training's reference columns
print("\n2. Processing TEST data...")
test_processed, test_metadata = preprocess_data(
    test_df_consistent,
    integer_encoding=all_integer_mappings,
    categorical_imputation_method='random',
    numerical_imputation_method='random',
    random_state=random_seed,
    reference_column_strategy='manual',
    manual_reference_columns=train_reference_cols,
    drop_reference_columns=True,
    dataset_prefix=DATASET_PREFIX,
)
print(f"   Test data processed: {test_processed.shape}")

# Process validation data - use 'manual' strategy with training's reference columns
print("\n3. Processing VALIDATION data...")
val_processed, val_metadata = preprocess_data(
    val_df_consistent,
    integer_encoding=all_integer_mappings,
    categorical_imputation_method='random',
    numerical_imputation_method='random',
    random_state=random_seed,
    reference_column_strategy='manual',
    manual_reference_columns=train_reference_cols,
    drop_reference_columns=True,
    dataset_prefix=DATASET_PREFIX,
)
print(f"   Validation data processed: {val_processed.shape}")

print(f"\nPreprocessing completed for all splits:")
print(f"  Training: {train_processed.shape}")
print(f"  Test: {test_processed.shape}")
print(f"  Validation: {val_processed.shape}")
print("=" * 70)

## Verify Preprocessing Results

In [ ]:
print("\n" + "=" * 70)
print("PREPROCESSING VERIFICATION")
print("=" * 70)

# Check column consistency
train_cols = set(train_processed.columns)
test_cols = set(test_processed.columns)
val_cols = set(val_processed.columns)

if train_cols == test_cols == val_cols:
    print("[ok] All datasets have identical columns")
    print(f"[ok] Total features: {len(train_cols)}")
else:
    error_msg = (
        f"Column mismatch: Train({len(train_cols)}), Test({len(test_cols)}), Val({len(val_cols)})"
    )
    only_in_train = train_cols - test_cols - val_cols
    only_in_test = test_cols - train_cols - val_cols
    only_in_val = val_cols - train_cols - test_cols
    if only_in_train:
        error_msg += f"\nOnly in train: {sorted(only_in_train)}"
    if only_in_test:
        error_msg += f"\nOnly in test: {sorted(only_in_test)}"
    if only_in_val:
        error_msg += f"\nOnly in val: {sorted(only_in_val)}"
    raise ValueError(error_msg)

# Verify encoding based on configuration
print(f"\nEncoding verification:")

# Extract integer-encoded columns from metadata (they no longer have _ordinal suffix)
integer_encoded_columns = [
    meta['created_columns'][0]
    for meta in train_metadata['encoding'].values()
    if meta['encoding_type'] == 'ordinal'
]

if INTEGER_ENCODING:
    print(f"Manual integer encoding configuration: Checking {len(INTEGER_ENCODING)} variables")
    for encoded_var in INTEGER_ENCODING.keys():
        # Column name is now the same as the original variable name
        if encoded_var in train_processed.columns:
            unique_vals = sorted(train_processed[encoded_var].unique())
            print(f"  [ok] {encoded_var}: {encoded_var} with values {unique_vals}")
        else:
            raise ValueError(
                f"Expected integer-encoded column '{encoded_var}' not found for variable '{encoded_var}'"
            )

elif FORCE_ALL_ORDINAL:
    print(f"Force all integer encoding mode: Checking {len(binary_vars + multi_cat_vars)} variables")
    for cat_col in (binary_vars + multi_cat_vars)[:10]:  # Show first 10
        if cat_col in train_processed.columns:
            unique_vals = sorted(train_processed[cat_col].unique())
            print(f"  [ok] {cat_col}: {cat_col} with values {unique_vals}")
        else:
            raise ValueError(
                f"Expected integer-encoded column '{cat_col}' not found for variable '{cat_col}'"
            )
    if len(binary_vars + multi_cat_vars) > 10:
        print(f"  ... and {len(binary_vars + multi_cat_vars) - 10} more")

else:
    print(f"default mode:")
    print(f"  Binary variables (integer encoded): {len(binary_vars)}")
    for binary_col in binary_vars[:5]:  # Show first 5
        if binary_col in train_processed.columns:
            unique_vals = sorted(train_processed[binary_col].unique())
            print(f"    [ok] {binary_col}: {binary_col} with values {unique_vals}")
    if len(binary_vars) > 5:
        print(f"    ... and {len(binary_vars) - 5} more")

    # Show one-hot encoded columns
    onehot_columns = [
        col
        for col in train_processed.columns
        if any(cat in col for cat in multi_cat_vars)
    ]
    print(f"  Multi-category variables (one-hot): {len(onehot_columns)} binary features")

print("=" * 70)

## Add ID Column and Save Datasets

In [ ]:
print("\n" + "=" * 70)
print("FINALIZING DATASETS")
print("=" * 70)

# Re-apply ID columns from original dataframe or create synthetic ones
if id_variable:
    print(f"Re-applying original ID column: {id_variable}")

    # Create mapping from index to original ID values
    # Note: The processed dataframes preserve indices from the original df_loaded
    try:
        id_mapping = pd.Series(df_loaded[id_variable].values, index=df_loaded.index)

        # Re-apply IDs to each split using preserved indices
        train_processed[id_variable] = train_processed.index.map(id_mapping)
        test_processed[id_variable] = test_processed.index.map(id_mapping)
        val_processed[id_variable] = val_processed.index.map(id_mapping)

        # Verify that all IDs were successfully mapped
        train_missing = train_processed[id_variable].isna().sum()
        test_missing = test_processed[id_variable].isna().sum()
        val_missing = val_processed[id_variable].isna().sum()

        if train_missing + test_missing + val_missing > 0:
            print(
                f"Warning: Missing ID values found - Train: {train_missing}, Test: {test_missing}, Val: {val_missing}"
            )
            print("This may indicate an issue with index preservation during processing.")
        else:
            print("[ok] All ID values successfully mapped to splits")

        # Additional verification: check for unique IDs within each split
        train_unique = train_processed[id_variable].nunique()
        test_unique = test_processed[id_variable].nunique()
        val_unique = val_processed[id_variable].nunique()

        print(f"ID uniqueness check:")
        print(
            f"  Train: {train_unique}/{len(train_processed)} unique ({train_unique == len(train_processed) and '[ok]' or '[!]'})"
        )
        print(
            f"  Test: {test_unique}/{len(test_processed)} unique ({test_unique == len(test_processed) and '[ok]' or '[!]'})"
        )
        print(
            f"  Val: {val_unique}/{len(val_processed)} unique ({val_unique == len(val_processed) and '[ok]' or '[!]'})"
        )

        print(f"ID column '{id_variable}' restored to all datasets")

    except Exception as e:
        print(f"Error mapping original IDs: {e}")
        print("Falling back to synthetic ID generation...")
        id_variable = None  # Fall through to synthetic ID creation

if not id_variable:
    print("No original ID column found or ID mapping failed, creating synthetic row IDs")

    # Generate unique IDs across all splits
    total_samples = len(train_processed) + len(test_processed) + len(val_processed)
    all_ids = [f"B{i:06d}" for i in range(total_samples)]

    # Assign IDs to each split
    train_end = len(train_processed)
    test_end = train_end + len(test_processed)

    train_processed['trr_id_code'] = all_ids[:train_end]
    test_processed['trr_id_code'] = all_ids[train_end:test_end]
    val_processed['trr_id_code'] = all_ids[test_end:]

    print("Created synthetic row IDs: trr_id_code")

# Save split datasets
print(f"\nSaving datasets to: {INTERIM_DATA_DIR}")
save_result = save_split_datasets(
    train_df=train_processed,
    test_df=test_processed,
    val_df=val_processed,
    base_filename=PROJECT_NAME,
    save_dir=INTERIM_DATA_DIR,
    include_timestamp=False,
    header_comment=None,
)

# Add output file hashes to training metadata
train_metadata['data_provenance']['output'] = {
    'train': save_result['train_hash_info'],
    'test': save_result['test_hash_info'],
    'val': save_result['val_hash_info'],
}

# Re-save metadata with output file hashes
from prism.preprocessing import _convert_to_json_serializable
import json
import glob

# Find the most recent metadata file for this dataset
metadata_pattern = str(PROCESSED_DATA_DIR / f"preprocessing_metadata_{DATASET_PREFIX}_*.json")
metadata_files = sorted(glob.glob(metadata_pattern), reverse=True)

if metadata_files:
    metadata_path = Path(metadata_files[0])
    print(f"Updating metadata with output file hashes: {metadata_path.name}")

    # Save updated metadata
    json_compatible_metadata = _convert_to_json_serializable(train_metadata)
    with open(metadata_path, 'w') as f:
        json.dump(json_compatible_metadata, f, indent=2)

    print("Metadata updated with output file hashes")
else:
    print("Warning: Could not find metadata file to update with output hashes")

print("Datasets saved successfully")
print("=" * 70)

## Target Variable Distribution Analysis

In [ ]:
if target_variable:
    print("\n" + "=" * 70)
    print("TARGET VARIABLE DISTRIBUTION ANALYSIS")
    print("=" * 70)
    print(f"Target variable: {target_variable}")

    # Training set
    print("\nTRAINING SET:")
    train_counts = train_processed[target_variable].value_counts().sort_index()
    train_props = train_processed[target_variable].value_counts(normalize=True).sort_index()
    for value in sorted(train_counts.index):
        count = train_counts[value]
        prop = train_props[value]
        print(f"  Class {value}: {count:5d} ({prop*100:5.2f}%)")

    # Test set
    print("\nTEST SET:")
    test_counts = test_processed[target_variable].value_counts().sort_index()
    test_props = test_processed[target_variable].value_counts(normalize=True).sort_index()
    for value in sorted(test_counts.index):
        count = test_counts[value]
        prop = test_props[value]
        print(f"  Class {value}: {count:5d} ({prop*100:5.2f}%)")

    # Validation set
    print("\nVALIDATION SET:")
    val_counts = val_processed[target_variable].value_counts().sort_index()
    val_props = val_processed[target_variable].value_counts(normalize=True).sort_index()
    for value in sorted(val_counts.index):
        count = val_counts[value]
        prop = val_props[value]
        print(f"  Class {value}: {count:5d} ({prop*100:5.2f}%)")

    # Summary comparison
    print("\nSUMMARY:")
    print("Dataset      | Class 0 | Class 1 | Total")
    print("-" * 45)
    print(
        f"Training     | {train_counts.get(0, 0):7d} | {train_counts.get(1, 0):7d} | {len(train_processed):5d}"
    )
    print(
        f"Test         | {test_counts.get(0, 0):7d} | {test_counts.get(1, 0):7d} | {len(test_processed):5d}"
    )
    print(
        f"Validation   | {val_counts.get(0, 0):7d} | {val_counts.get(1, 0):7d} | {len(val_processed):5d}"
    )
    print(
        f"Total        | {train_counts.get(0, 0) + test_counts.get(0, 0) + val_counts.get(0, 0):7d} | {train_counts.get(1, 0) + test_counts.get(1, 0) + val_counts.get(1, 0):7d} | {len(train_processed) + len(test_processed) + len(val_processed):5d}"
    )

    # Check for class imbalance
    print("\nCLASS IMBALANCE:")
    for dataset_name, df in [
        ("Training", train_processed),
        ("Test", test_processed),
        ("Validation", val_processed),
    ]:
        props = df[target_variable].value_counts(normalize=True).sort_index()
        if len(props) == 2:
            minority_class = props.min()
            majority_class = props.max()
            imbalance_ratio = majority_class / minority_class
            print(
                f"  {dataset_name:12}: Minority={minority_class:.4f} ({minority_class*100:.2f}%), Ratio={imbalance_ratio:.2f}:1"
            )

    print("=" * 70)
else:
    print("\nNo target variable detected - skipping distribution analysis")

## Label File Generation

In [ ]:
# Import the label file template generator from prism
from prism.feature_labels import generate_label_file_template

print("\n" + "=" * 70)
print("LABEL FILE GENERATION")
print("=" * 70)

# Determine the example_notebooks directory
current_dir = Path.cwd()

if (current_dir / "example_notebooks").exists():
    example_notebooks_dir = current_dir / "example_notebooks"
elif current_dir.name == "example_notebooks":
    example_notebooks_dir = current_dir
else:
    # Try to find the example_notebooks directory
    example_notebooks_dir = None
    for parent in current_dir.parents:
        potential_dir = parent / "example_notebooks"
        if potential_dir.exists():
            example_notebooks_dir = potential_dir
            break

    if not example_notebooks_dir:
        example_notebooks_dir = current_dir
        print("Warning: Could not find example_notebooks directory. Using current directory.")

# Define file paths
main_label_file = example_notebooks_dir / "generic_variable_labels.csv"
template_label_file = example_notebooks_dir / "generic_variable_labels_template.csv"

print(f"Label file directory: {example_notebooks_dir}")
print(f"Main label file: {main_label_file.name}")

# Check if main label file exists
if main_label_file.exists():
    print(f"\nMain label file already exists: {main_label_file.name}")

    # Create timestamped template instead
    timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
    timestamped_template = (
        example_notebooks_dir / f"generic_variable_labels_template_{timestamp}.csv"
    )

    print(f"Creating timestamped template: {timestamped_template.name}")
    label_template = generate_label_file_template(
        df_loaded, train_processed, target_variable, id_variable, timestamped_template, train_metadata
    )

    print("This preserves your existing labels while providing an updated template.")
    print("\nNext steps:")
    print("1. Compare the new template with your existing label file")
    print("2. Update your template label file with any new variables if needed")
    print(
        f"3. Save as '{DATASET_PREFIX}_variable_labels.csv' for this dataset (recommended)"
    )
    print(
        "   OR as 'generic_variable_labels.csv' for reuse across datasets with matching variable names"
    )
    print(
        f"\nNote: Dataset-specific files (e.g., '{DATASET_PREFIX}_variable_labels.csv') take priority."
    )
    print(
        "      Generic files only provide labels for variables that exist in the current dataset."
    )
else:
    print(f"\nGenerating new label file...")
    label_template = generate_label_file_template(
        df_loaded, train_processed, target_variable, id_variable, main_label_file, train_metadata
    )

    print(f"\nLabel file saved to: {main_label_file}")
    print("\nNext steps:")
    print("1. Open the generated label file")
    print("2. Fill in the 'user_label' column with meaningful descriptions")
    print(
        f"3. Save as '{DATASET_PREFIX}_variable_labels.csv' for this dataset (recommended)"
    )
    print(
        "   OR keep as 'generic_variable_labels.csv' for reuse with datasets that have matching variable names"
    )
    print(
        f"\nNote: Dataset-specific files (e.g., '{DATASET_PREFIX}_variable_labels.csv') take priority."
    )
    print(
        "      Generic files only provide labels for variables that exist in the current dataset."
    )

print("=" * 70)

## Summary

In [ ]:
print("\n" + "=" * 70)
print("PREPROCESSING PIPELINE COMPLETE")
print("=" * 70)

print(f"\nDataset: {DATASET_PREFIX}")
print(f"  Original: {df_loaded.shape[0]} rows, {df_loaded.shape[1]} columns")
print(f"  Final processed datasets:")
print(f"    Training: {train_processed.shape[0]} rows, {train_processed.shape[1]} columns")
print(f"    Test: {test_processed.shape[0]} rows, {test_processed.shape[1]} columns")
print(f"    Validation: {val_processed.shape[0]} rows, {val_processed.shape[1]} columns")

print(f"\nConfiguration:")
print(f"  Target variable: {target_variable if target_variable else 'Not detected'}")
print(f"  ID variable: {id_variable if id_variable else 'Generated synthetic IDs (trr_id_code)'}")
print(f"  Splitting method: {SPLITTING_METHOD}")
if SPLITTING_METHOD == "temporal":
    print(f"  Time variable: {TIME_VARIABLE}")

print(f"\nCategorical encoding:")
if INTEGER_ENCODING:
    print(f"  Strategy: Manual ordinal configuration")
    print(f"  Ordinal features: {len(INTEGER_ENCODING)}")
elif FORCE_ALL_ORDINAL:
    print(f"  Strategy: Force all categorical → ordinal")
    print(f"  Ordinal features: {len(all_integer_mappings)}")
else:
    print(f"  Strategy: default (Binary → integer, Multi-category → one-hot)")
    print(f"  Binary variables (ordinal): {len(binary_vars)}")
    print(f"  Multi-category variables (one-hot): {len(multi_cat_vars)}")

if SELECTED_FEATURES:
    print(f"\nFeature selection:")
    print(f"  Selected features: {len(SELECTED_FEATURES)}")

print(f"\nOutput files:")
print(f"  Location: {INTERIM_DATA_DIR}")
print(f"  Files: {PROJECT_NAME}_train.csv, {PROJECT_NAME}_test.csv, {PROJECT_NAME}_val.csv")
print(f"  Label file template: {main_label_file.name}")
print(
    f"    Recommended: Save as '{DATASET_PREFIX}_variable_labels.csv' (dataset-specific, higher priority)"
)

print(f"\nNext steps:")
print(f"  1. Review and fill in the label file template")
print(f"     Save as: {DATASET_PREFIX}_variable_labels.csv (recommended)")
print(f"     Or as: generic_variable_labels.csv (for reuse with datasets that have matching variables)")
print(f"  2. Run a modeling notebook (e.g., train_mlp.ipynb)")
print(f"  3. Use PRiSM for model interpretation")

print("=" * 70)